<a href="https://colab.research.google.com/github/minxuan0923/AML_Assignment/blob/mx/Set5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import libraries

In [50]:
import pandas as pd
from scipy.io import arff
import requests
from io import StringIO

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, fbeta_score

import tensorflow as tf
from tensorflow.keras import datasets, layers, models, optimizers, losses
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt

## Load Dataset

In [51]:
URL = (
    "https://raw.githubusercontent.com/"
    "minxuan0923/AML_Assignment/main/data/"
    "UCI_Phishing_Training_Dataset.arff"
)

response = requests.get(URL, timeout=30)
response.raise_for_status()

data, meta = arff.loadarff(StringIO(response.text))

df = pd.DataFrame(data)

for col in df.select_dtypes([object]):
    df[col] = pd.to_numeric(df[col].str.decode('utf-8'))

X_data = df.drop(columns =['Result'])
y_data = df['Result']

## Spliting

In [52]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_data, y_data, random_state = 42, test_size=0.3, stratify=y_data)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


## Encoding

In [53]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)

# One-hot-encoding X values
X_train_encoded = encoder.fit_transform(X_train)
X_val_encoded = encoder.fit_transform(X_val)
X_test_encoded = encoder.transform(X_test)

# Convert y values - phishing to "1" and legitimate to "0"
y_train = y_train.replace({-1: 1, 1: 0})
y_val = y_val.replace({-1: 1, 1: 0})
y_test = y_test.replace({-1: 1, 1: 0})

## Autoencoder

In [55]:

class Autoencoder(Model):
  def __init__(self, latent_dim, shape):
    super(Autoencoder, self).__init__()
    self.latent_dim = latent_dim
    self.shape = shape
    self.encoder = tf.keras.Sequential([
      layers.Flatten(),
      layers.Dense(latent_dim, activation='relu'),
    ])
    self.decoder = tf.keras.Sequential([
      layers.Dense(int(tf.math.reduce_prod(shape)), activation='sigmoid'),
      layers.Reshape(shape)
    ])

  def call(self, x):
    encoded = self.encoder(x)
    decoded = self.decoder(encoded)
    return decoded


shape = X_test_encoded.shape[1:]
latent_dim = 16 # we will experiment with 4, 8, 16, 32
autoencoder = Autoencoder(latent_dim, shape)

autoencoder.compile(optimizer='adam', loss=losses.binary_crossentropy)

history_ae = autoencoder.fit(X_train_encoded, X_train_encoded,
                epochs=10,
                shuffle=True,
                validation_data=(X_val_encoded, X_val_encoded))

X_train_compressed = autoencoder.encoder.predict(X_train_encoded)
X_val_compressed = autoencoder.encoder.predict(X_val_encoded)
X_test_compressed = autoencoder.encoder.predict(X_test_encoded)

Epoch 1/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.4788 - val_loss: 0.3458
Epoch 2/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.2909 - val_loss: 0.2483
Epoch 3/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2130 - val_loss: 0.1830
Epoch 4/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1621 - val_loss: 0.1451
Epoch 5/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1311 - val_loss: 0.1200
Epoch 6/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1100 - val_loss: 0.1030
Epoch 7/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0955 - val_loss: 0.0911
Epoch 8/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0856 - val_loss: 0.0831
Epoch 9/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0788 - val_loss: 0.0773
Epoch 10/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0739 - val_loss: 0.0738
242/242 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  


## Hyperparameter Grid

In [62]:
learning_rates = [0.001, 0.01, 0.1]
batch_sizes = [16, 32, 64]
epochs_list = [50, 100, 150]

## Logistic Regression

In [63]:
def build_logistic_model(learning_rate):

    model = models.Sequential([
        layers.Input(shape=(X_train_compressed.shape[1],)),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


## Grid Search

In [67]:
def evaluate_model(model, X, y):
    y_prob = model.predict(X, verbose=0)
    y_pred = (y_prob >= 0.5).astype(int).flatten()
    results = {
        'Accuracy': accuracy_score(y, y_pred),
        'Precision': precision_score(y, y_pred, zero_division=0),
        'Recall': recall_score(y, y_pred, zero_division=0),
        'F1-score': f1_score(y, y_pred, zero_division=0),
        'F2-score': fbeta_score(y, y_pred, beta=2),
    }

    return results

In [66]:
grid_results = []
for lr in learning_rates:
    for batch in batch_sizes:
        for epoch in epochs_list:

            # Reset model state and random seed
            tf.keras.backend.clear_session()
            tf.keras.utils.set_random_seed(42)

            # Build a new model
            model = build_logistic_model(lr)

            # Train the model
            model.fit(
                X_train_compressed,
                y_train,
                batch_size=batch,
                epochs=epoch,
                verbose=0
            )

            # Evaluate using validation set
            results = evaluate_model(
                model,
                X_val_compressed,
                y_val
            )

            # Record hyperparameters
            results['Learning Rate'] = lr
            results['Batch Size'] = batch
            results['Epochs'] = epoch

            # Store the results
            grid_results.append(results)

            print(
                f"LR: {lr}, Batch: {batch}, "
                f"Epochs: {epoch}, "
                f"F2-score: {results['F2-score']:.4f}"
            )


LR: 0.001, Batch: 16, Epochs: 50, F2-score: 0.9121
LR: 0.001, Batch: 16, Epochs: 100, F2-score: 0.9107
LR: 0.001, Batch: 16, Epochs: 150, F2-score: 0.9096
LR: 0.001, Batch: 32, Epochs: 50, F2-score: 0.9121
LR: 0.001, Batch: 32, Epochs: 100, F2-score: 0.9118


KeyboardInterrupt: 

## Display Results

In [ ]:
df_results = pd.DataFrame(grid_results)

df_results = df_results.sort_values(
    by='F2-score',
    ascending=False
).reset_index(drop=True)

display(df_results)